DPA sanity check for SDP model outputs in Gulf Stream area.

In [ ]:
from typing import cast
from matplotlib.figure import Figure
from matplotlib.axes import Axes
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from glob import glob

data_dir = "/Users/jerry/school/research/eddy-tracking/data/gulf_stream_20240305_20260531/silver"

In [ ]:
def load_pigments(polarity):
    fps = glob(f"{data_dir}/pigments/{polarity}/*.parquet")
    dfs = [pd.read_parquet(fp) for fp in fps]
    combined = pd.concat(dfs, ignore_index=True)
    combined["polarity"] = polarity
    return combined

df = pd.concat(
    [load_pigments("cyclone"), load_pigments("anticyclone")],
    ignore_index=True,
)
print(f"pixel_observations: {len(df):,}")
df["polarity"].value_counts()

In [ ]:
# Uitz et al. (2006) DPA coefficients
dpa_weights = {
    "Fuco": 1.41, "Perid": 1.41, "HexFuco": 1.27,
    "ButFuco": 0.35, "Allo": 0.60, "MV chlb": 1.01, "Zea": 0.86,
}

df["dpa_tchla"] = sum(w * df[pig] for pig, w in dpa_weights.items())

residual = df["dpa_tchla"] - df["T chla"]
r2 = 1 - (residual ** 2).sum() / ((df["T chla"] - df["T chla"].mean()) ** 2).sum()
rmse = np.sqrt((residual ** 2).mean())
bias = residual.mean()
print(
    f"r2: {r2:.4f}\n"
    f"rmse_ug_l: {rmse:.4f}\n"
    f"bias_ug_l: {bias:.4f}"
)

In [ ]:
cyc = df[df["polarity"] == "cyclone"]
anti = df[df["polarity"] == "anticyclone"]

fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(7, 7)),
)
ax.scatter(cyc["T chla"], cyc["dpa_tchla"], s=4, alpha=0.3, c="#1f77b4", label="Cyclone")
ax.scatter(anti["T chla"], anti["dpa_tchla"], s=4, alpha=0.3, c="#d62728", label="Anticyclone")

lims = [0, max(df["T chla"].max(), df["dpa_tchla"].max()) * 1.05]
ax.plot(lims, lims, "k--", lw=1, label="1:1")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("SDP TChla (µg/L)")
ax.set_ylabel("DPA TChla (µg/L)")
ax.set_title("DPA vs SDP TChla")
ax.legend(markerscale=3)
plt.tight_layout()

In [ ]:
fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(7, 5)),
)
bins = np.linspace(-1.5, 1.5, 80)
ax.hist(cyc["dpa_tchla"] - cyc["T chla"], bins=bins, alpha=0.6, color="#1f77b4", label="Cyclone")
ax.hist(anti["dpa_tchla"] - anti["T chla"], bins=bins, alpha=0.6, color="#d62728", label="Anticyclone")
ax.axvline(0, color="k", ls="--", lw=1)
ax.set_xlabel("Residual (DPA − SDP)")
ax.set_ylabel("Count")
ax.set_title("Residual distribution")
ax.legend()
plt.tight_layout()